In [ ]:
#| default_exp unigram

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re, os, sys, mmap, struct, zlib, hashlib, tempfile, subprocess, importlib.util, tokenizers, numpy as np
from array import array
from contextlib import contextmanager
from functools import lru_cache
from pathlib import Path
from tokenizers import Tokenizer, PreTokenizedString
from fastcore.all import ifnone
from laghu.core import log, find_vocab, is_unigram, ODD
try: import fcntl
except ImportError: fcntl = None
try: import laghu_rs
except ImportError: laghu_rs = None

## Why

A Unigram tokenizer has a vocabulary of pieces, each with a score. To tokenize a word it considers every way of splitting the word into pieces from the vocabulary and picks the split whose scores add up highest. The search that does this is a Viterbi search: it walks along the word and, at each position, keeps the best split of everything before it.

The `tokenizers` library speeds up the lookups with a character trie over all the pieces. For potion-multilingual-128M's 500,353 pieces that trie takes about 610 MB of private memory in every process. laghu instead keeps the pieces in a file it builds once per tokenizer.json: a [marisa trie](https://github.com/pytries/marisa-trie) of the pieces, with their ids and scores beside it. Every process maps that file read-only and shares its pages, so the vocabulary costs a process almost no private memory. When the file cannot be used, because `shared=False` was asked for, marisa-trie is not installed, or there is nowhere to write the file, the pieces go into a Python dict, which costs about 80 MB per process. The normalizer, which cleans up the text, and the pre-tokenizer, which first splits it into words, still come from `tokenizers`, built from the same tokenizer.json with the vocabulary removed.

## Reading the vocabulary

`read_vocab(fn)` returns the vocabulary of a Unigram tokenizer.json as a dict from piece to id, an array of scores, and the rest of the file as parsed JSON with the vocabulary emptied out. It reads the pieces one at a time from the memory-mapped file. Parsing the whole file as JSON, or even reading it into one bytes object first, leaves about 250 MB behind that Python's allocator keeps after the parse. When a piece appears twice it keeps the later id, as `tokenizers` does.

In [ ]:
#| export
_ENTRY = re.compile(rb'\s*\[\s*"((?:[^"\\]|\\.)*)"\s*,\s*([^\],\s]+)\s*\]\s*(,?)')
_CLOSE = re.compile(rb'\s*\]')

def _vocab(t, fn):
    "The pieces (piece → id), scores and the vocabulary's span in tokenizer.json `fn`'s bytes `t`; a repeated piece keeps its last id, as in `tokenizers`."
    v0, br = find_vocab(t)
    if br != b'[': raise NotImplementedError(f'{fn}: not a Unigram vocabulary')
    ids, scores, at, i = {}, array('d'), v0, 0
    while m := _ENTRY.match(t, at):
        p = m.group(1); ids[json.loads(b'"' + p + b'"') if b'\\' in p else p.decode()] = i; scores.append(float(m.group(2))); i += 1; at = m.end()
        if not m.group(3): break
    if not (c := _CLOSE.match(t, at)): raise ValueError(f'{fn}: vocabulary not read to its end (stopped at byte {at})')
    return ids, scores, v0, c.start()

def read_vocab(fn):
    "The vocabulary of Unigram tokenizer.json `fn` as `(piece -> id, scores, the JSON without its vocabulary)`."
    with open(fn, 'rb') as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as t:
        ids, scores, v0, v1 = _vocab(t, fn)
        return ids, scores, json.loads(t[:v0] + t[v1:])

## Segmenting a pre-token

A pre-token is one of the chunks the pre-tokenizer produces, usually a word. `viterbi(s, ids, scores, reach, unk_id, unk_score)` segments one pre-token `s` and returns the ids of its pieces. It is a port of `Unigram::encode_optimized` from Hugging Face's `tokenizers`, and follows it exactly so that the ids come out the same:

- At each position it keeps the best-scoring path that ends there. A new candidate replaces it only when its score is strictly higher, so a tie keeps the path found first.
- A character that is not itself a piece is taken as unknown, with the score `unk_score`, which laghu sets to the lowest score in the vocabulary minus `UNK_PENALTY` (10).
- Consecutive unknown characters are joined into one unknown token, as `tokenizers` does with its `fuse_unk` option. `_unk` then decides what to emit for the joined string: its id if it happens to be a piece, else its UTF-8 bytes as `<0xXX>` pieces when the model has byte fallback, else the unknown id. A model with no unknown id raises `ValueError`, as `tokenizers` does.

`reach` maps the first two characters of each piece to the length of the longest piece that starts with them, so the search never tries a substring longer than any piece could be. The example uses a five-piece vocabulary.

In [ ]:
#| export
UNK_PENALTY = 10.0
WORD_CACHE, WORD_LEN = 2**16, 32   # pre-tokens cached per Unigram, and the longest one cached

def _unk(p, ids, unk_id, byte_fallback):
    "The ids of an unknown run `p`: a piece if it spells one, else its UTF-8 bytes as `<0xXX>` pieces under byte fallback, else the unknown id."
    if (k := ids.get(p)) is not None: return [k]
    if byte_fallback and None not in (bs := [ids.get(f'<0x{b:02X}>') for b in p.encode()]): return bs
    if unk_id < 0: raise ValueError(f'Encountered an unknown token but `unk_id` is missing: {p!r}')   # tokenizers' error, and model2vec's
    return [unk_id]

def _path(s, start, tid, ids, unk_id, byte_fallback):
    "The ids on the best path, walked back from the end of `s`, each run of unknowns fused and spelled by `_unk`."
    out, e, run = [], len(s), None
    while e > 0:
        if tid[e] == unk_id: run = run or e
        else:
            if run: out += _unk(s[e:run], ids, unk_id, byte_fallback)[::-1]; run = None
            out.append(tid[e])
        e = start[e]
        if run and e == 0: out += _unk(s[:run], ids, unk_id, byte_fallback)[::-1]
    return out[::-1]

def viterbi(s, ids, scores, reach, unk_id, unk_score, byte_fallback=False):
    "The ids of the highest-scoring split of pre-token `s` into pieces of `ids`."
    n = len(s)
    best, start, tid = [0.0] + [None] * n, [0] * (n + 1), [0] * (n + 1)
    for i in range(n):
        b, single = best[i], False
        for j in range(i + 1, min(n, i + reach.get(s[i:i + 2], 1)) + 1):
            k = ids.get(s[i:j])
            if k is None: continue
            c = scores[k] + b
            if best[j] is None or c > best[j]: best[j], start[j], tid[j] = c, i, k
            if j == i + 1: single = True
        if not single:
            c = unk_score + b
            if best[i + 1] is None or c > best[i + 1]: best[i + 1], start[i + 1], tid[i + 1] = c, i, unk_id
    return _path(s, start, tid, ids, unk_id, byte_fallback)

In [ ]:
ids = {'a': 0, 'b': 1, 'ab': 2, '<unk>': 3, 'xy': 4}
sc = array('d', [-1.0, -1.0, -1.5, -5.0, -2.0])
reach = dict(ab=2, xy=2)
v = lambda s: viterbi(s, ids, sc, reach, 3, min(sc) - UNK_PENALTY)
assert v('ab') == [2] and v('ba') == [1, 0] and v('') == []
assert v('aqqb') == [0, 3, 1]          # the two unknown letters are one token
assert v('xy') == [4] and v('xq') == [3] and v('qxy') == [3, 4]
bids = {**ids, '<0x71>': 5, '<0xC3>': 6, '<0xA9>': 7}
bv = lambda s: viterbi(s, bids, array('d', [*sc, -1, -1, -1]), reach, 3, min(sc) - UNK_PENALTY, True)
assert bv('aqqb') == [0, 5, 5, 1] and bv('é') == [6, 7] and bv('ä') == [3]   # 'ä' is 0xC3 0xA4, and there is no <0xA4>

## A shared vocabulary

`write_vocab(fn, p)` turns a tokenizer.json's vocabulary into one binary file at `p`. The file holds a header, the piece ids and their float64 scores in the trie's own key order, a count of pieces by length (used for `median_token_length`), the marisa trie itself, and a byte trie of the same pieces for the optional Rust core, behind its own small header (`_CSR`), with every array at an 8-byte boundary. The header records a magic number, the format version, the counts, the size and modification time of the tokenizer.json it came from, where the vocabulary sits in that file, the lowest score, and a CRC-32 checksum of everything after the header. `Vocab(p, fn)` maps the file read-only and checks all of this. It raises `ValueError` when the file is too short, damaged, of another version, or older than its tokenizer.json, and `shared_vocab` then builds it again. `Vocab.get(piece)` looks up a piece's id, and `Vocab.dict()` returns the whole vocabulary as `read_vocab` would.

`shared_vocab(fn)` is the function to call: it returns a `Vocab` for a tokenizer.json, building the file if it is missing, stale or damaged, or None when it cannot. The file goes in `cache_dir()`, which is `$LAGHU_CACHE`, else `$XDG_CACHE_HOME/laghu`, else `~/.cache/laghu`, under a name derived from the tokenizer.json's resolved path (`vocab_paths`), so laghu never writes into a model folder, a git checkout or the Hugging Face cache. When the cache cannot be written, the file goes next to tokenizer.json, and when neither works, `Unigram` falls back to the dict. The file is written under a temporary name and renamed into place, so a reader never sees half of it. When several processes start at once, one takes a lock and builds the file, in a child process so that its own memory never holds the half-million strings of the build, while the others wait and then map what it wrote.

`viterbi_trie(s, v, unk_id, unk_score)` is `viterbi` over a `Vocab`. At each position, one prefix search of the trie returns every piece that starts there, where the dict version needs one lookup for each possible length. The candidates are the same pieces with the same float64 scores, compared in the same order, so the ids are the same, and the prefix search is about three times as fast. The cell below builds vocabulary files for two small tokenizers and checks that `viterbi_trie` agrees with `viterbi`.

In [ ]:
#| export
VOCAB_MAGIC, VOCAB_VERSION = b'LAGHUVOC', 2
_HDR = struct.Struct('<8sIIIiQQQQQdI')   # magic, version, pieces in the trie, longest piece, id of '' or -1, source size and mtime_ns, vocabulary span, trie bytes, lowest score, CRC-32 of the rest
_CSR = struct.Struct('<8sII')   # magic, byte-trie nodes, byte-trie edges
CSR_MAGIC, NO_VALUE = b'LAGHUCSR', 0xFFFFFFFF

def cache_dir():
    "The folder for vocabulary files: `$LAGHU_CACHE`, else `$XDG_CACHE_HOME/laghu`, else `~/.cache/laghu`."
    if p := os.environ.get('LAGHU_CACHE'): return Path(p)
    return Path(os.environ.get('XDG_CACHE_HOME') or Path.home()/'.cache')/'laghu'

def vocab_paths(fn):
    "The places the vocabulary file of tokenizer.json `fn` may go, in order: `cache_dir()`, then next to `fn`."
    h = hashlib.blake2b(str(Path(fn).resolve()).encode(), digest_size=8).hexdigest()
    return [cache_dir()/f'{Path(fn).stem}-{h}.laghu', Path(fn).with_suffix('.laghu')]

def _up(x): return -(-x // 8) * 8

def _layout(nk, longest):
    "Byte offsets of the ids, scores, length counts and trie in a vocabulary file of `nk` pieces, the longest `longest` long."
    ids = _up(_HDR.size); sc = _up(ids + 4 * nk); hist = sc + 8 * nk
    return ids, sc, hist, hist + 8 * (longest + 1)

def _csr_layout(at, nodes, edges):
    "Byte offsets of the byte trie's header, edge starts, child nodes, node values and edge bytes, and of its end, for a section from `at`."
    h = _up(at); st = _up(h + _CSR.size); ch = _up(st + 4 * (nodes + 1)); va = _up(ch + 4 * edges); eb = _up(va + 4 * nodes)
    return h, st, ch, va, eb, eb + edges

def csr_trie(keys):
    "The byte trie of `keys`, `(bytes, value)` pairs sorted by bytes, as arrays: each node's first edge, edge bytes, child nodes, node values."
    par, byt, val, path, prev = [], [], [NO_VALUE], [0], b''
    for key, k in keys:
        l, m = 0, min(len(prev), len(key))
        while l < m and prev[l] == key[l]: l += 1
        del path[l + 1:]
        for b in key[l:]: par.append(path[-1]); byt.append(b); path.append(len(val)); val.append(NO_VALUE)
        val[path[-1]] = k; prev = key
    par = np.array(par, np.int64); o = np.argsort(par, kind='stable')   # sorted keys create each node's children in byte order
    start = np.zeros(len(val) + 1, '<u4'); start[1:] = np.cumsum(np.bincount(par, minlength=len(val)))
    return start, np.array(byt, np.uint8)[o], (o + 1).astype('<u4'), np.array(val, '<u4')

def write_vocab(fn, p):
    "Build the vocabulary file for tokenizer.json `fn` and write it to `p`."
    import marisa_trie
    st = os.stat(fn)
    with open(fn, 'rb') as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as t: ids, scores, v0, v1 = _vocab(t, fn)
    ks = [k for k in ids if k]
    tr = marisa_trie.Trie(ks); nk, tb = len(tr), tr.tobytes()
    es, eb, ec, va = csr_trie(sorted((k.encode(), i) for k, i in tr.iteritems()))
    vid = np.zeros(nk, np.int32); vid[[tr[k] for k in ks]] = [ids[k] for k in ks]
    lens = np.fromiter(map(len, ids), np.int64, len(ids)); longest = int(lens.max(initial=0))
    o = _layout(nk, longest); c = _csr_layout(o[3] + len(tb), len(va), len(eb)); b = bytearray(c[-1])
    b[o[0]:o[0] + 4 * nk] = vid.astype('<i4').tobytes()
    b[o[1]:o[1] + 8 * nk] = np.frombuffer(scores, np.float64)[vid].astype('<f8').tobytes()
    b[o[2]:o[3]] = np.bincount(lens, minlength=longest + 1).astype('<i8').tobytes()
    b[o[3]:o[3] + len(tb)] = tb
    _CSR.pack_into(b, c[0], CSR_MAGIC, len(va), len(eb))
    for at, a in zip(c[1:5], (es, ec, va, eb)): b[at:at + a.nbytes] = a.tobytes()
    _HDR.pack_into(b, 0, VOCAB_MAGIC, VOCAB_VERSION, nk, longest, ids.get('', -1), st.st_size, st.st_mtime_ns, v0, v1, len(tb), min(scores),
                   zlib.crc32(memoryview(b)[_HDR.size:]))
    p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=p.name + '.', suffix='.tmp', dir=p.parent)
    try:
        with os.fdopen(fd, 'wb') as f: f.write(b)
        os.chmod(tmp, 0o644); os.replace(tmp, p)
    except BaseException: Path(tmp).unlink(missing_ok=True); raise
    log.info(f'laghu: wrote {p} for {fn}')

class Vocab:
    "A vocabulary file mapped read-only and shared between processes, holding the pieces, their ids and their scores."
    def __init__(self, p, fn):
        "Map file `p`, raising `ValueError` when it is short, damaged, of another version, or older than tokenizer.json `fn`."
        import marisa_trie
        with open(p, 'rb') as f: mm, self.ino = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ), os.fstat(f.fileno()).st_ino
        if len(mm) < _HDR.size: raise ValueError(f'{p}: truncated')
        mg, ver, nk, self.longest, self.empty, sz, mt, v0, v1, tl, self.lowest, crc = _HDR.unpack_from(mm)
        if (mg, ver) != (VOCAB_MAGIC, VOCAB_VERSION): raise ValueError(f'{p}: not a version {VOCAB_VERSION} laghu vocabulary')
        st = os.stat(fn)
        if (sz, mt) != (st.st_size, st.st_mtime_ns): raise ValueError(f'{p}: stale, {fn} has changed')
        o = _layout(nk, self.longest); c0 = _up(o[3] + tl)
        if len(mm) < c0 + _CSR.size: raise ValueError(f'{p}: truncated')
        cm, nodes, edges = _CSR.unpack_from(mm, c0)
        if cm != CSR_MAGIC: raise ValueError(f'{p}: no byte trie')
        c = _csr_layout(o[3] + tl, nodes, edges)
        if len(mm) != c[-1]: raise ValueError(f'{p}: {len(mm):,} bytes where its header makes {c[-1]:,}')
        if zlib.crc32(memoryview(mm)[_HDR.size:]) != crc: raise ValueError(f'{p}: checksum mismatch')
        with open(fn, 'rb') as f, mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as t:
            if t[v0 - 1:v0] != b'[' or not _CLOSE.match(t, v1): raise ValueError(f'{p}: its vocabulary span is not {fn}\'s')
            self.cfg = json.loads(t[:v0] + t[v1:])   # tokenizer.json without its vocabulary
        self.mm, mv = mm, memoryview(mm)
        self.ids, self.scores = mv[o[0]:o[0] + 4 * nk].cast('i'), mv[o[1]:o[1] + 8 * nk].cast('d')
        self.hist = np.frombuffer(mm, '<i8', self.longest + 1, o[2])
        self.trie = marisa_trie.Trie(); self.trie.map(mv[o[3]:o[3] + tl])
        self.csr = (mv[c[1]:c[1] + 4 * (nodes + 1)].cast('I'), mv[c[4]:c[5]], mv[c[2]:c[2] + 4 * edges].cast('I'), mv[c[3]:c[3] + 4 * nodes].cast('I'))

    def __len__(self): return len(self.trie) + (self.empty >= 0)

    def get(self, p):
        "The id of piece `p`, or None."
        if not p: return self.empty if self.empty >= 0 else None
        k = self.trie.get(p)
        return None if k is None else self.ids[k]

    def dict(self):
        "The whole vocabulary as a dict from piece to id, in id order."
        d = {k: self.ids[i] for k, i in sorted(self.trie.iteritems(), key=lambda o: self.ids[o[1]])}
        if self.empty >= 0: d[''] = self.empty
        return d

@contextmanager
def _lock(p):
    "An exclusive lock on `p`'s lock file, held while one process checks and builds `p`."
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p.with_name(p.name + '.lock'), 'a') as f:
        if fcntl: fcntl.flock(f, fcntl.LOCK_EX)
        yield

def _spawn(fn, p):
    "`write_vocab(fn, p)` in a child process, so this one's heap never holds the half-million strings of a build."
    subprocess.run([sys.executable, '-c', 'import sys; from laghu.unigram import write_vocab; write_vocab(*sys.argv[1:])', str(fn), str(p)],
                   check=True, capture_output=True, timeout=600)

def shared_vocab(fn):
    "The `Vocab` for tokenizer.json `fn`, built when missing, stale or damaged, or None when no file can be had."
    if sys.byteorder != 'little' or not importlib.util.find_spec('marisa_trie'): return None
    for p in vocab_paths(fn):
        try: return Vocab(p, fn)
        except (OSError, ValueError): pass
        try:
            with _lock(p):
                try: return Vocab(p, fn)   # another process built it while this one waited
                except (OSError, ValueError): pass
                try: _spawn(fn, p); return Vocab(p, fn)
                except (OSError, ValueError, subprocess.SubprocessError): write_vocab(fn, p); return Vocab(p, fn)   # no child: build here
        except Exception as e: log.info(f'laghu: no vocabulary file at {p}: {e!r}')

def viterbi_trie(s, v, unk_id, unk_score, byte_fallback=False):
    "`viterbi` over the mapped vocabulary `v`, using one trie prefix search per position."
    n, L, pre, ids, sc = len(s), v.longest, v.trie.iter_prefixes_with_ids, v.ids, v.scores
    best, start, tid = [0.0] + [None] * n, [0] * (n + 1), [0] * (n + 1)
    for i in range(n):
        b, single = best[i], False
        for p, k in pre(s[i:i + L]):
            j = i + len(p); c = sc[k] + b
            if best[j] is None or c > best[j]: best[j], start[j], tid[j] = c, i, ids[k]
            if j == i + 1: single = True
        if not single:
            c = unk_score + b
            if best[i + 1] is None or c > best[i + 1]: best[i + 1], start[i + 1], tid[i + 1] = c, i, unk_id
    return _path(s, start, tid, v, unk_id, byte_fallback)

In [ ]:
import tempfile, subprocess
from fastcore.all import Path

In [ ]:
tmp = Path(tempfile.mkdtemp()); os.environ['LAGHU_CACHE'] = str(tmp/'cache')

def tiny(fn, vocab, unk_id=3, byte_fallback=False):
    "A minimal Unigram tokenizer.json at `fn`."
    d = dict(version='1.0', truncation=None, padding=None, added_tokens=[], normalizer=None, pre_tokenizer=None, post_processor=None, decoder=None,
             model=dict(type='Unigram', unk_id=unk_id, vocab=vocab, byte_fallback=byte_fallback))
    Path(fn).write_text(json.dumps(d)); return Path(fn)

voc = [['a', -1.0], ['b', -1.0], ['ab', -1.5], ['<unk>', -5.0], ['xy', -2.0]]
tf = tiny(tmp/'tiny.json', voc)
v = shared_vocab(tf); p = vocab_paths(tf)[0]
assert isinstance(v, Vocab) and p.exists() and len(v) == 5 and v.get('ab') == 2 and v.get('q') is None and v.dict() == read_vocab(tf)[0]
for s in ['ab', 'ba', '', 'aqqb', 'xy', 'xq', 'qxy']:
    assert viterbi_trie(s, v, 3, v.lowest - UNK_PENALTY) == viterbi(s, ids, sc, reach, 3, min(sc) - UNK_PENALTY), s
bf = tiny(tmp/'bytes.json', [*voc, ['<0x71>', -1.0], ['<0xC3>', -1.0], ['<0xA9>', -1.0]], byte_fallback=True)
bvt = lambda s: viterbi_trie(s, shared_vocab(bf), 3, -15.0, True)
assert bvt('aqqb') == [0, 5, 5, 1] and bvt('é') == [6, 7] and bvt('ä') == [3]

The file also holds a second index of the same pieces, for the optional Rust core `laghu_rs`: a byte trie in four flat arrays, built by `csr_trie` from the pieces' UTF-8 bytes. Node 0 is the root. Node `n`'s edges run from `start[n]` to `start[n + 1]` in the edge bytes, which are sorted within a node, and in the child nodes, and its value is the piece's position in the ids and scores, or `NO_VALUE` when no piece ends there. `Vocab.csr` maps the four arrays from the file, so a process that searches them holds no private copy of them. With `laghu_rs` installed, its `Seg.word` gives `viterbi_trie`'s ids, byte fallback included. The arrays of the five-piece vocabulary above:

In [ ]:
if laghu_rs:
    s = laghu_rs.Seg(*v.csr, v.ids, v.scores, 3, v.lowest - UNK_PENALTY, False); vb = shared_vocab(bf)
    assert all(s.word(w) == viterbi_trie(w, v, 3, v.lowest - UNK_PENALTY) for w in ['ab', 'ba', '', 'aqqb', 'xy', 'xq', 'qxy'])
    assert [laghu_rs.Seg(*vb.csr, vb.ids, vb.scores, 3, -15.0, True).word(w) for w in ('aqqb', 'é', 'ä')] == [[0, 5, 5, 1], [6, 7], [3]]
dict(zip(['start', 'ebyte', 'echild', 'val'], (a.tolist() for a in v.csr)))

A second call maps the file it finds and builds nothing. A changed tokenizer.json, a truncated file, a flipped byte and a file of another version are each detected and rebuilt. A model in a read-only folder gets its file in the cache, and with nowhere at all to write, `shared_vocab` returns None.

In [ ]:
st0 = p.stat()
assert shared_vocab(tf).ino == st0.st_ino and p.stat().st_mtime_ns == st0.st_mtime_ns   # mapped, not rebuilt
tiny(tf, [*voc, ['qq', -0.5]])
assert shared_vocab(tf).get('qq') == 5 and p.stat().st_ino != st0.st_ino   # stale: rebuilt
def spoil(f):
    b = bytearray(p.read_bytes()); f(b); p.write_bytes(b)
    assert (v := shared_vocab(tf)) and v.get('qq') == 5, f
for f in [lambda b: b.__delitem__(slice(-7, None)), lambda b: b.__setitem__(-3, b[-3] ^ 1), lambda b: b.__setitem__(8, 99),
          lambda b: b.__delitem__(slice(20, None)), lambda b: b.clear()]:
    spoil(f)
ro = tmp/'ro'; ro.mkdir(); rf = tiny(ro/'tokenizer.json', voc); ro.chmod(0o555)
assert shared_vocab(rf) and not (ro/'tokenizer.laghu').exists()   # a read-only model folder: the file is in the cache
os.environ['LAGHU_CACHE'] = str(ro/'cache')                        # nowhere to write
assert shared_vocab(rf) is None
os.environ['LAGHU_CACHE'] = str(tmp/'cache'); ro.chmod(0o755)

laghu 0.2 wrote version 1 of the file: the same header and sections, without the byte trie. `as_v1` turns a file back into that, byte for byte what laghu 0.2.2 writes from the same tokenizer.json (checked once on potion-multilingual-128M's 7.6 MB file), and the next load builds version 2 again, as it does for any file of another version:

In [ ]:
def as_v1(p):
    "Rewrite vocabulary file `p` as laghu 0.2 wrote it: version 1, without the byte trie."
    b = bytearray(p.read_bytes()); hd = list(_HDR.unpack_from(b))
    b = b[:_layout(hd[2], hd[3])[3] + hd[9]]; hd[1] = 1; hd[-1] = zlib.crc32(memoryview(b)[_HDR.size:]); _HDR.pack_into(b, 0, *hd)
    p.write_bytes(b)

ver = lambda p: _HDR.unpack_from(p.read_bytes())[1]
as_v1(p); before = ver(p)
assert shared_vocab(tf).get('qq') == 5
(before, ver(p))

## The tokenizer

`Unigram(fn, shared=True)` gives the same ids as `tokenizers`' `Tokenizer.encode(text, add_special_tokens=False).ids` for a Unigram tokenizer.json. Call it with a list of texts to get one list of ids per text. It works in four steps:

1. It finds the added tokens, special strings such as `[PAD]`, `<extra_id_0>` or XLM-R's `<mask>` that always map to one id, and cuts them out of the raw text, trying the longest match first. A token marked `lstrip` or `rstrip` also takes the whitespace on that side of it. `Added` does this matching.
2. Each stretch of text between them goes through the normalizer. Added tokens marked `normalized` are matched after normalization instead; one model's `[PAD]`, for example, becomes `[ PAD ]` once normalized.
3. The pre-tokenizer splits what is left into pre-tokens, and `viterbi` segments each one. A pre-token of up to `WORD_LEN` (32) characters is first looked up in a cache of up to `WORD_CACHE` (65,536) pre-tokens per tokenizer, because natural text repeats its words. The result of `viterbi` depends only on the pre-token, so the cache can never change an id. Some models, potion-multilingual-128M among them, use a Metaspace pre-tokenizer with `split: false`, which makes the whole text a single pre-token; for those the cache helps only with a text seen before.
4. Truncation and padding, when tokenizer.json asks for them, are applied to each batch as `encode_batch` applies them. model2vec embeds the padding ids along with the rest.

One detail needs care. A Metaspace pre-tokenizer marks the start of words with `▁`, and with `prepend_scheme: first` it adds a `▁` only where the normalized text starts at the very beginning of the original text. If the normalizer removes a leading space or byte-order mark, no `▁` is added. The first stretch is therefore processed through a `PreTokenizedString`, which keeps track of how the normalized text lines up with the original, and the stretches after an added token use a copy of the pre-tokenizer with the scheme set to `never`.

Two kinds of tokenizer raise `NotImplementedError`, and `tokenizer` then falls back to `tokenizers`. One has `single_word` added tokens, because Rust decides what counts as a word with Unicode's Alphabetic property, which Python does not provide. The other has `normalized` added tokens under `prepend_scheme: first`.

With the optional wheel `laghu-rs` installed (module `laghu_rs`, built from `rust/` in this repository), step 3's segmenting moves to Rust. `split(s)` gives steps 1 to 3's pre-tokens and added-token ids, and `rs`, a `laghu_rs.Seg` over the mapped byte trie and the mapped ids and scores, segments a whole batch on `threads` threads (all cores by default) with the GIL released. It runs the same search over UTF-8 byte offsets: the same float64 sums, the candidates in the same order, the same strict comparison, the same handling of unknown runs and byte fallback, so the ids are the same. `rs` is None, and the Python search is used as before, when `laghu_rs` is not installed, when `LAGHU_RUST=0` is set as the `Unigram` is made, when there is no shared file, or when tokenizer.json truncates or pads.

Steps 1 to 3 move to Rust as well, so a whole batch runs in the threads from the raw texts to the ids. `rs_pre`, a `laghu_rs.Pre`, holds a `Tokenizer` from the `tokenizers` crate built from the same tokenizer.json with the one-piece model `_pipeline` uses, but with the added tokens kept, and it splits a text as `Tokenizer.encode` does before the model runs: `AddedVocabulary::extract_and_normalize`, then the pre-tokenizer. The crate is pinned to the version of Python `tokenizers` laghu matches, so this is the same code that model2vec's ids come from. `rs_texts` gives `rs` the texts whole when `rs_pre` is set, and as `split` gives them when it is None. `rs_batch` and `rs_encode` pass a batch through `rs_texts` to `rs`. `rs_pre` is None when the installed `tokenizers` is another version, when the crate refuses the tokenizer.json, or when its split differs from `split`'s on any of `PROBE`, the built-in difficult texts under 10,000 characters.

`encode(s)` gives the ids of one text before truncation and padding, always by the Python search. `vocab()` returns the vocabulary as `Tokenizer.get_vocab()` does, `n` is its size and `median` the median piece length, and `unk_token_id` is the id model2vec removes from every encoding, which for a Unigram model is None.

In [ ]:
#| export
PROBE = [s for s in ODD if len(s) < 10_000]   # texts `Unigram._rs_pre` and `HFTok._rust` check laghu_rs on, a few milliseconds' worth
WS = frozenset('\t\n\x0b\x0c\r \x85\xa0\u1680\u2000\u2001\u2002\u2003\u2004\u2005\u2006\u2007\u2008\u2009\u200a\u2028\u2029\u202f\u205f\u3000')   # Rust's char::is_whitespace

def _metaspace(pt, scheme):
    "Pre-tokenizer JSON `pt` with every Metaspace's `prepend_scheme: first` set to `scheme`."
    if isinstance(pt, list): return [_metaspace(o, scheme) for o in pt]
    if not isinstance(pt, dict): return pt
    pt = {k: _metaspace(v, scheme) for k, v in pt.items()}
    if pt.get('type') == 'Metaspace' and pt.get('prepend_scheme') == 'first': pt['prepend_scheme'] = scheme
    return pt

def compatible(a, b):
    "Whether `tokenizers` versions `a` and `b` share their major and minor number; `PROBE` then decides."
    return bool(a and b) and str(a).split('.')[:2] == str(b).split('.')[:2]

def _stub(d):
    "Tokenizer JSON `d` with a one-piece model and nothing after pre-tokenization: its normalizer, pre-tokenizer and added tokens."
    d = dict(d, post_processor=None, decoder=None, truncation=None, padding=None)
    d['model'] = dict(d['model'], vocab=[['<unk>', 0.0]], unk_id=0, byte_fallback=False)
    return d

def _pipeline(d, pre):
    "The normalizer and pre-tokenizer of tokenizer JSON `d`, with pre-tokenizer `pre`, and nothing else of it."
    t = Tokenizer.from_str(json.dumps(dict(_stub(d), pre_tokenizer=pre, added_tokens=[])))
    return t.normalizer, t.pre_tokenizer, t

class Added:
    "Finds added tokens in a string the way `tokenizers`' `AddedVocabulary.find_matches` does."
    def __init__(self, toks, norm=None):
        "Match the added tokens `toks` as written, or as normalized by `norm` when it is given."
        self.toks = {(norm.normalize_str(o['content']) if norm else o['content']): (o['id'], o.get('lstrip'), o.get('rstrip')) for o in toks}
        self.toks.pop('', None)
        self.rx = re.compile('|'.join(map(re.escape, sorted(self.toks, key=len, reverse=True)))) if self.toks else None

    def split(self, s):
        "Cut `s` into `(text, None)` and `(token, id)` parts, in order."
        if not self.rx: return [(s, None)] if s else []
        out, prev = [], 0
        for m in self.rx.finditer(s):
            st, sp = m.span(); k, ls, rs = self.toks[m.group()]
            if ls:
                while st > prev and s[st - 1] in WS: st -= 1
            if rs:
                while sp < len(s) and s[sp] in WS: sp += 1
            if st > prev: out.append((s[prev:st], None))
            out.append((s[st:sp], k)); prev = sp
        if prev < len(s): out.append((s[prev:], None))
        return out

class Unigram:
    "A Unigram tokenizer that gives the same ids as `tokenizers`' `encode_batch(texts, add_special_tokens=False)`."
    def __init__(self, fn, shared=True):
        "Load tokenizer.json `fn`, mapping the vocabulary from a shared file when `shared` is set and possible, else using a dict."
        self.mapped = shared_vocab(fn) if shared else None
        if self.mapped is not None: d = self.mapped.cfg
        else: self.ids, self.scores, d = read_vocab(fn)
        mdl = d['model']
        if mdl.get('type', 'Unigram') != 'Unigram': raise NotImplementedError(f"{fn}: model is {mdl.get('type')}")
        added = d.get('added_tokens', [])
        if any(o.get('single_word') for o in added): raise NotImplementedError(f'{fn}: single_word added tokens')
        self.byte_fallback, self.unk_id = bool(mdl.get('byte_fallback')), ifnone(mdl.get('unk_id'), -1)   # -1: none, an error if met
        self.trunc, self.pad = d.get('truncation'), d.get('padding')
        self.unk_score = (self.mapped.lowest if self.mapped is not None else min(self.scores)) - UNK_PENALTY
        if self.mapped is None:
            self.reach = {}   # a piece's first two letters → the longest piece that starts so: no longer substring is tried
            for w in self.ids:
                if len(w) > 1 and len(w) > self.reach.get(w[:2], 0): self.reach[w[:2]] = len(w)
        self.added = {o['content']: o['id'] for o in added}
        self.norm, self.pre, t = _pipeline(d, d.get('pre_tokenizer'))
        self.raw, self.nrm = Added([o for o in added if not o.get('normalized')]), Added([o for o in added if o.get('normalized')], self.norm)
        later = _metaspace(d.get('pre_tokenizer'), 'never')
        self.pre_later = self.pre if later == d.get('pre_tokenizer') else _pipeline(d, later)[1]
        if self.nrm.rx and self.pre_later is not self.pre: raise NotImplementedError(f'{fn}: normalized added tokens under prepend_scheme first')
        self.model_unk = getattr(t.model, 'unk_token', None)   # model2vec drops this token's id; a Unigram model has none
        pieces = self.ids if self.mapped is None else self.mapped
        extra = [k for k in self.added if pieces.get(k) is None]
        self.n = len(pieces) + len(extra)   # the size of `Tokenizer.get_vocab()`
        h = self.mapped.hist if self.mapped is not None else np.bincount(np.fromiter(map(len, self.ids), np.int64, len(self.ids)), minlength=1)
        lens = np.concatenate([np.repeat(np.arange(len(h)), h), np.fromiter(map(len, extra), np.int64, len(extra))])
        self.median = int(np.median(lens)) if self.n else 0
        self._cached = lru_cache(WORD_CACHE)(lambda q: tuple(self._viterbi(q)))
        self.rs, self.threads = self._rust(), os.cpu_count() or 1
        self.rs_pre = self._rs_pre(d) if self.rs is not None else None

    def _rust(self):
        "A `laghu_rs.Seg` over the mapped file, or None: no extension, `LAGHU_RUST=0`, no shared file, or truncation or padding to apply."
        if laghu_rs is None or os.environ.get('LAGHU_RUST') == '0' or self.mapped is None or self.trunc or self.pad: return None
        v = self.mapped
        return laghu_rs.Seg(*v.csr, v.ids, v.scores, self.unk_id, self.unk_score, self.byte_fallback)

    def _rs_pre(self, d):
        "A `laghu_rs.Pre` for tokenizer JSON `d`, or None: a `tokenizers` of another minor version than the one it was built with, or a split unlike `split`'s."
        if not compatible(getattr(laghu_rs, 'TOKENIZERS', None), tokenizers.__version__): return None
        try: pre = laghu_rs.Pre(json.dumps(_stub(d)), {k: v for k, v in self.added.items() if k})
        except ValueError as e: log.info('laghu_rs cannot split for this tokenizer: %s', e); return None
        if any(pre.split(s) != self.split(s) for s in PROBE): log.warning('laghu_rs splits text unlike tokenizers; splitting in Python'); return None
        return pre

    def rs_texts(self, txts):
        "The texts `rs` takes for `txts`, and the `laghu_rs.Pre` that splits them: whole when `rs_pre` is set, else as `split` gives them."
        return (list(txts), self.rs_pre) if self.rs_pre is not None else ([self.split(t) for t in txts], None)

    def rs_batch(self, txts, drop=None, max_len=None):
        "The ids of each text in `txts` by `rs`, id `drop` removed and cut to `max_len`."
        xs, pre = self.rs_texts(txts); return self.rs.batch(xs, drop, max_len, self.threads, pre)

    def rs_encode(self, txts, table, out, drop=None, max_len=None):
        "`rs_batch`, then each text's mean row of float32 `table` written into float32 `out`; returns each text's count of ids."
        xs, pre = self.rs_texts(txts); return self.rs.encode(xs, table, out, drop, max_len, self.threads, pre)

    def _viterbi(self, q):
        if self.mapped is not None: return viterbi_trie(q, self.mapped, self.unk_id, self.unk_score, self.byte_fallback)
        return viterbi(q, self.ids, self.scores, self.reach, self.unk_id, self.unk_score, self.byte_fallback)

    def word(self, q):
        "The ids of pre-token `q`, cached when it is at most `WORD_LEN` characters long."
        return self._cached(q) if len(q) <= WORD_LEN else self._viterbi(q)

    def _first(self, s):
        "The pre-tokens of the stretch opening the text, offsets kept, so `first` prepends only where the normalized text starts at 0."
        pts = PreTokenizedString(s)
        if self.norm: pts.normalize(self.norm.normalize)
        pts.split(lambda i, ns: [ns[0:len(ns.normalized)]] if ns.normalized else [])   # as tokenizers re-slices after its added-token pass
        self.pre.pre_tokenize(pts)
        return [p for p, *_ in pts.get_splits()]

    def _words(self, s, first):
        "Stretch `s`, free of raw added tokens, as pre-tokens (str) and normalized added-token ids (int)."
        if first and self.pre_later is not self.pre: return self._first(s)
        nz = self.norm.normalize_str(s) if self.norm else s
        if not nz: return []
        out, pt = [], (self.pre if first else self.pre_later)
        for p, k in self.nrm.split(nz):
            if k is not None: out.append(k)
            else: out += [q for q, _ in pt.pre_tokenize_str(p)] if pt else [p]
        return out

    def split(self, s):
        "Text `s` as the Viterbi search receives it: pre-tokens (str) to segment, and the ids (int) of added tokens."
        out, at = [], 0
        for p, k in self.raw.split(s):
            if k is None: out += self._words(p, at == 0)
            else: out.append(k)
            at += len(p)
        return out

    def encode(self, s):
        "The ids of text `s`, before truncation and padding, by the Python Viterbi search."
        return [k for q in self.split(s) for k in (self.word(q) if type(q) is str else [q])]

    def __call__(self, txts):
        "The ids of each text in `txts` as one batch, by `laghu_rs` when `rs` is set, truncated and padded when tokenizer.json asks for it."
        if self.rs is not None: return self.rs_batch(txts)
        out = [self.encode(t) for t in txts]
        if t := self.trunc:
            L, left = t['max_length'], t.get('direction') == 'Left'
            out = [(o[-L:] if L else []) if left else o[:L] for o in out]
        if (p := self.pad) and out:
            st = p.get('strategy')
            n = max(map(len, out)) if st == 'BatchLongest' else (st or {}).get('Fixed', 0)
            if (m := p.get('pad_to_multiple_of')) and n % m: n += m - n % m
            pid, left = p.get('pad_id', 0), p.get('direction') == 'Left'
            out = [([pid] * (n - len(o)) + o if left else o + [pid] * (n - len(o))) if len(o) < n else o for o in out]
        return out

    @property
    def unk_token_id(self):
        "The id model2vec drops from every encoding, or None."
        return None if self.model_unk is None else self.vocab()[self.model_unk]

    def vocab(self):
        "The vocabulary as `Tokenizer.get_vocab()` gives it, added tokens included."
        return {**(self.ids if self.mapped is None else self.mapped.dict()), **self.added}

## Everything else

`HFTok(fn)` wraps `tokenizers` itself, for WordPiece, BPE and WordLevel models and for any Unigram tokenizer that `Unigram` refuses. It has the same attributes as `Unigram` (`n`, `median`, `unk_token_id`, `vocab()`), which are the ones model2vec reads from a tokenizer, so `LeanModel` treats the two alike.

With `laghu_rs` installed, `HFTok` tokenizes a BERT WordPiece tokenizer.json in Rust: one whose normalizer is a BertNormalizer, whose pre-tokenizer is a BertPreTokenizer and whose model is WordPiece. `rs` is then a `laghu_rs.WordPiece`, a port of `FastWordPiece` from model2vec-rs. It does what `encode_single_sequence` does, without the offsets and alignments that `tokenizers` keeps. It matches the added tokens in the raw text, longest first. It normalizes the text between them with the Unicode tables the `tokenizers` crate uses, splits it into words, and segments each word by WordPiece's greedy longest match. `rs` is None when `laghu_rs` is not installed or has no `WordPiece`, when `LAGHU_RUST=0` is set as the `HFTok` is made, when `rust=False`, when tokenizer.json truncates or pads, when the installed `tokenizers` is another minor version, when `laghu_rs.WordPiece` refuses the file, or when its ids differ from `tokenizers`' on any of `PROBE`. `rs_batch` and `rs_encode` call `rs` for `LeanModel`, as `Unigram`'s do.

`tokenizer(fn, lean=True, shared=True)` chooses between them. It returns a `Unigram` for a Unigram tokenizer.json, with its vocabulary in a shared file when `shared` is set, and an `HFTok` otherwise, or when `lean=False`, or when `Unigram` refuses the file. An `HFTok` made with `lean=False` never uses `laghu_rs`.

In [ ]:
#| export
class HFTok:
    "A `tokenizers` tokenizer with the attributes model2vec reads, encoding a BERT WordPiece pipeline by `laghu_rs` when it can."
    def __init__(self, fn, rust=True):
        "Load tokenizer.json `fn`, with a `laghu_rs.WordPiece` as `rs` when `rust` is set and `_rust` allows it."
        self.tok = Tokenizer.from_file(str(fn))
        v = self.tok.get_vocab()
        self.n, self.median = len(v), int(np.median([len(o) for o in v])) if v else 0
        unk = getattr(self.tok.model, 'unk_token', None)
        self.unk_token_id = v[unk] if unk is not None else None
        self._enc = getattr(self.tok, 'encode_batch_fast', self.tok.encode_batch)
        self.rs, self.threads = self._rust(fn) if rust else None, os.cpu_count() or 1

    def _rust(self, fn):
        "A `laghu_rs.WordPiece` for tokenizer.json `fn`, or None: no extension or one without it, `LAGHU_RUST=0`, truncation or padding to apply, a `tokenizers` of another minor version, a pipeline it does not port, or ids unlike `tokenizers`' on `PROBE`."
        if getattr(laghu_rs, 'WordPiece', None) is None or os.environ.get('LAGHU_RUST') == '0' or self.tok.truncation or self.tok.padding: return None
        if not isinstance(self.tok.model, tokenizers.models.WordPiece): return None
        if not compatible(laghu_rs.TOKENIZERS, tokenizers.__version__): return None
        try: wp = laghu_rs.WordPiece(Path(fn).read_text(encoding='utf-8'))
        except ValueError as e: log.info('laghu_rs cannot tokenize this tokenizer: %s', e); return None
        if wp.batch(PROBE) != self._hf(PROBE): log.warning('laghu_rs.WordPiece gives ids unlike tokenizers; tokenizing with tokenizers'); return None
        return wp

    def _hf(self, txts): return [e.ids for e in self._enc(list(txts), add_special_tokens=False)]

    def rs_batch(self, txts, drop=None, max_len=None):
        "The ids of each text in `txts` by `rs`, id `drop` removed and cut to `max_len`."
        return self.rs.batch(list(txts), drop, max_len, self.threads)

    def rs_encode(self, txts, table, out, drop=None, max_len=None):
        "`rs_batch`, then each text's mean row of float32 `table` written into float32 `out`; returns each text's count of ids."
        return self.rs.encode(list(txts), table, out, drop, max_len, self.threads)

    def __call__(self, txts):
        "The ids of each text in `txts`, encoded as one batch by `rs` when it is set, else by `tokenizers`."
        return self.rs_batch(txts) if self.rs is not None else self._hf(txts)

    def vocab(self): return self.tok.get_vocab()

def tokenizer(fn, lean=True, shared=True):
    "A `Unigram` for a Unigram tokenizer.json `fn` when `lean` is set and it supports the file, else an `HFTok`, with its Rust path when `lean` is set."
    if lean and is_unigram(fn):
        try: return Unigram(fn, shared)
        except NotImplementedError as e: log.info(f'laghu: {e}; tokenizing with tokenizers')
    return HFTok(fn, lean)

## Tests

`Unigram` must give the same ids as `tokenizers.encode_batch` on the test corpus, in batches of 64 so that padding to the longest text in a batch is exercised. potion-multilingual-128M uses bge-m3's tokenizer, with Metaspace `always` and two added tokens. t5-small's tokenizer.json has no model type, uses WhitespaceSplit before Metaspace and has 103 added tokens. Five published distilled models, and variants of potion's tokenizer.json, cover the remaining features. `same` reports the first text where the ids differ.

In [ ]:
from laghu.core import try_resolve, model_files, corpus, ODD
from fastcore.all import Path
import tempfile

@contextmanager
def no_rust():
    "`LAGHU_RUST=0` while it is held, so a `Unigram` made then takes the Python path."
    os.environ['LAGHU_RUST'] = '0'
    try: yield
    finally: del os.environ['LAGHU_RUST']

def phase1(fn, shared=True):
    "`Unigram(fn, shared)` segmenting in Rust what Python's `split` gives, as laghu_rs did before it could split."
    u = Unigram(fn, shared); u.rs_pre = None; return u

def same(fn, txts, bs=64, shared=True):
    "Whether `Unigram(fn, shared)` gives `tokenizers`' ids on every one of `txts`: splitting and segmenting in Rust when it can, segmenting only, and in Python; the first difference is printed."
    t, us = Tokenizer.from_file(str(fn)), [Unigram(fn, shared), phase1(fn, shared)]
    with no_rust(): us.append(Unigram(fn, shared))
    assert all((u.mapped is not None) == shared for u in us) and us[2].rs is None
    assert us[0].rs is None or us[0].rs_pre is not None   # wherever Rust segments, it also splits
    for u, nm in zip(us, ('rust', 'rust, python split', 'python')):
        for i in range(0, len(txts), bs):
            b = txts[i:i + bs]
            for s, a, e in zip(b, u(b), t.encode_batch(b, add_special_tokens=False)):
                if a != e.ids: print(nm, repr(s[:80]), a[:20], e.ids[:20]); return False
    return True

def hub_file(repo, fn):
    "One file of `repo` from the Hub cache or downloaded, or None offline."
    from huggingface_hub import hf_hub_download
    try: return Path(hf_hub_download(repo, fn))
    except Exception as e: print(f'skipping {repo}: {e!r}')

TOKS = ['<mask>', '<s>', '</s>', '<pad>', '[PAD]', '[UNK]', '<unk>', '<extra_id_0>', '<bos>', '<eos>', 'Computer Science']
txts = corpus(1500) + [f'{a}{t}{b}' for t in TOKS for a in ('', 'x', 'x ', ' x  ', '\u3000') for b in ('', 'y', ' y', '  \n')]
tmp = Path(tempfile.mkdtemp()); os.environ['LAGHU_CACHE'] = str(tmp/'cache')

In [ ]:
ro.chmod(0o555); os.environ['LAGHU_CACHE'] = str(ro/'cache')   # with nowhere to write the vocabulary goes in a dict
assert Unigram(rf).mapped is None and Unigram(rf).encode('aqqb') == [0, 3, 1]
ro.chmod(0o755); os.environ['LAGHU_CACHE'] = str(tmp/'cache')
assert Unigram(rf).mapped is not None and Unigram(rf).encode('aqqb') == [0, 3, 1] and Unigram(rf, shared=False).mapped is None

In [ ]:
if f := try_resolve('minishlab/potion-multilingual-128M'):
    fn = model_files(f).tok; vp = vocab_paths(fn)[0]; shared_vocab(fn); as_v1(vp)   # as laghu 0.2 left it: rebuilt as version 2 by the next load
    assert same(fn, txts) and ver(vp) == VOCAB_VERSION and same(fn, txts, shared=False)
    with no_rust(): u = Unigram(fn)
    a = u(txts); d = Unigram(fn, shared=False)
    assert (u.n, u.median, u.vocab()) == (d.n, d.median, d.vocab()) and list(u.vocab()) == list(d.vocab())
    assert u(txts) == a and u._cached.cache_info().hits > 0   # a second pass comes from the cache, unchanged
    assert Unigram(fn).rs is not None or laghu_rs is None

Four processes that start at the same moment with an empty cache build the file once: one holds the lock and builds it, and the others wait and map what it wrote. A fifth process maps the same file and writes nothing.

In [ ]:
_PROBE = 'import sys; from laghu.unigram import shared_vocab; v = shared_vocab(sys.argv[1]); print(v.ino if v else 0)'
probe = lambda fn: subprocess.Popen([sys.executable, '-c', _PROBE, str(fn)], stdout=subprocess.PIPE, text=True)
if f := try_resolve('minishlab/potion-multilingual-128M'):
    fn = model_files(f).tok; os.environ['LAGHU_CACHE'] = str(tmp/'race')
    inos = {int(p.communicate()[0]) for p in [probe(fn) for _ in range(4)]}
    vf = vocab_paths(fn)[0]; st = vf.stat()
    assert inos == {st.st_ino}, inos
    assert int(probe(fn).communicate()[0]) == st.st_ino and vf.stat().st_mtime_ns == st.st_mtime_ns and not list(vf.parent.glob('*.tmp'))
    os.environ['LAGHU_CACHE'] = str(tmp/'cache')

In [ ]:
if fn := hub_file('google-t5/t5-small', 'tokenizer.json'):
    assert is_unigram(fn) and 'type' not in json.loads(fn.read_text())['model']
    assert same(fn, txts)

jina-embeddings-v3 uses XLM-R's tokenizer, with a `<mask>` token marked `lstrip`, and WhitespaceSplit before Metaspace. The EmbeddingGemma distill has byte fallback, a Replace normalizer and 255,000 pieces. The Spanish intent model uses ByteLevel pre-tokenizing and `normalized` added tokens, with truncation and fixed padding at 512. The Qwen3 distill uses ByteLevel over a Unigram vocabulary. A Danish model is tested too.

In [ ]:
for repo in ['CISCai/jina-embeddings-v3-separation-distilled', 'mikeee/m2v-gemma-embedding-300m', 'OpenVoiceOS/ovos-m2v-intents-es-ES-bne-v5',
             'tss-deposium/m2v-qwen3-embedding-0.6b-1024d', 'andersborges/model2vecdk']:
    if fn := hub_file(repo, 'tokenizer.json'): assert isinstance(tokenizer(fn), Unigram), repo; assert same(fn, txts), repo

Variants of potion-multilingual-128M's tokenizer.json, each changing one thing: `prepend_scheme` set to `first` and to `never`; no normalizer; no pre-tokenizer; `[PAD]` with `lstrip` and `[UNK]` with `rstrip`; both of those `normalized`; byte fallback over the pieces `<0x00>` to `<0xFF>`; padding to the longest text, on the left, to a multiple of 8; and truncation to 16 ids from the left.

In [ ]:
def variant(nm, ch, src=None):
    d2 = json.loads(json.dumps(ifnone(src, d))); ch(d2); (tmp/f'{nm}.json').write_text(json.dumps(d2, ensure_ascii=False)); return tmp/f'{nm}.json'

def bytes_(d):
    v = d['model']['vocab']; v += [[f'<0x{b:02X}>', -20.0] for b in range(256)]; d['model']['byte_fallback'] = True

def flag(**kw):
    def f(d): [o.update(kw.get(o['content'], {})) for o in d['added_tokens']]
    return f

if f := try_resolve('minishlab/potion-multilingual-128M'):
    d = json.loads(Path(model_files(f).tok).read_text())
    for nm, ch in [('first', lambda d: d['pre_tokenizer'].update(prepend_scheme='first')),
                   ('never', lambda d: d['pre_tokenizer'].update(prepend_scheme='never')),
                   ('nonorm', lambda d: d.update(normalizer=None)), ('nopre', lambda d: d.update(pre_tokenizer=None)),
                   ('strip', flag(**{'[PAD]': dict(lstrip=True), '[UNK]': dict(rstrip=True)})),
                   ('normed', flag(**{'[PAD]': dict(normalized=True), '[UNK]': dict(normalized=True, lstrip=True)})),
                   ('bytes', bytes_),
                   ('pad', lambda d: d.update(padding=dict(strategy='BatchLongest', direction='Left', pad_to_multiple_of=8, pad_id=0, pad_type_id=0, pad_token='[PAD]'))),
                   ('trunc', lambda d: d.update(truncation=dict(direction='Left', max_length=16, strategy='LongestFirst', stride=0)))]:
        assert same(variant(nm, ch), txts), nm

When `laghu_rs` cannot split as `split` does, `Unigram` keeps `rs` but splits in Python, as before it could split. `_rs_pre` is the check: it returns None for a tokenizer.json the `tokenizers` crate refuses, and for one whose split differs from `split`'s on the probe texts, here a normalizer that lowercases where the model's does not. It also returns None when the installed `tokenizers` is not the version `laghu_rs` was built with, because model2vec's ids come from the installed one.

In [ ]:
fell = None
if laghu_rs and (f := try_resolve('minishlab/potion-multilingual-128M')):
    u = Unigram(model_files(f).tok)
    refused, unlike = dict(d, normalizer={'type': 'NoSuchNormalizer'}), dict(d, normalizer={'type': 'Lowercase'})
    assert u.rs_pre is not None and u._rs_pre(d) is not None
    fell = [u._rs_pre(refused), u._rs_pre(unlike)]
    assert fell == [None, None]
fell

In [ ]:
assert compatible('0.23.2', '0.23.3') and not compatible('0.23.2', '0.22.2') and not compatible('0.23.2', '1.0.0') and not compatible(None, '0.23.2')
kept = None
if laghu_rs and (f := try_resolve('minishlab/potion-multilingual-128M')):
    import tokenizers as _tk
    u, real = Unigram(model_files(f).tok), _tk.__version__
    a, b = laghu_rs.TOKENIZERS.split('.')[:2]
    try:
        # another patch release of the same minor keeps the Rust split, as long as PROBE agrees; another minor does not
        _tk.__version__ = f'{a}.{b}.99'; kept = u._rs_pre(d) is not None
        _tk.__version__ = f'{a}.{int(b) - 1}.0'; dropped = u._rs_pre(d) is None
    finally: _tk.__version__ = real
    assert kept and dropped
kept

Finally, the fallback. A WordPiece tokenizer.json is not Unigram, and `Unigram` refuses a file with a `single_word` added token, so `tokenizer` returns an `HFTok` for both.

In [ ]:
if f := try_resolve('minishlab/potion-base-8M'):
    assert isinstance(tokenizer(model_files(f).tok), HFTok)
if f := try_resolve('minishlab/potion-multilingual-128M'):
    assert isinstance(tokenizer(variant('sw', flag(**{'[PAD]': dict(single_word=True)}))), HFTok) and isinstance(tokenizer(model_files(f).tok), Unigram)

With `laghu_rs` installed, `HFTok` tokenizes potion-base-8M's BERT WordPiece tokenizer.json in Rust. Each variant changes one setting: no lowercasing, accents stripped without lowercasing, accents kept with lowercasing, CJK ideographs not spaced, no cleaning, words of more than 5 characters made unknown, and an added token that is not special. Every one gives `tokenizers`' ids on all of `txts` and `ODD`, in batches of 64. A normalized added token, an `lstrip` added token, padding and a normalizer inside a `Sequence` each leave `rs` None, and so does `lean=False`:

In [ ]:
def same_wp(fn, txts, bs=64):
    "Whether `HFTok(fn)` gives `tokenizers`' ids on every one of `txts` by `laghu_rs.WordPiece`; the first difference is printed."
    h, t = HFTok(fn), Tokenizer.from_file(str(fn))
    assert h.rs is not None, fn
    for i in range(0, len(txts), bs):
        b = txts[i:i + bs]
        for s, a, e in zip(b, h(b), t.encode_batch(b, add_special_tokens=False)):
            if a != e.ids: print(repr(s[:80]), a[:20], e.ids[:20]); return False
    return True

def norm(**kw): return lambda d: d['normalizer'].update(kw)

wp = None
if laghu_rs and (f := try_resolve('minishlab/potion-base-8M')):
    dw = json.loads(Path(model_files(f).tok).read_text())
    extra = dict(content='Computer Science', single_word=False, lstrip=False, rstrip=False, normalized=False, special=False)
    kept = [('wp', lambda d: None), ('cased', norm(lowercase=False)), ('accents', norm(lowercase=False, strip_accents=True)),
            ('keepaccents', norm(strip_accents=False)), ('nochinese', norm(handle_chinese_chars=False)), ('noclean', norm(clean_text=False)),
            ('short', lambda d: d['model'].update(max_input_chars_per_word=5)),
            ('added', lambda d: d['added_tokens'].append(dict(extra, id=len(d['model']['vocab']))))]
    wp = {nm: same_wp(variant(nm, ch, dw), txts + ODD) for nm, ch in kept}
    assert all(wp.values()), wp
    refused = [('wpnormed', flag(**{'[MASK]': dict(normalized=True)})), ('wplstrip', flag(**{'[MASK]': dict(lstrip=True)})),
               ('wppad', lambda d: d.update(padding=dict(strategy='BatchLongest', direction='Right', pad_to_multiple_of=None, pad_id=0, pad_type_id=0, pad_token='[PAD]'))),
               ('wpseq', lambda d: d.update(normalizer=dict(type='Sequence', normalizers=[d['normalizer']])))]
    assert all(HFTok(variant(nm, ch, dw)).rs is None for nm, ch in refused)
    assert tokenizer(model_files(f).tok).rs is not None and tokenizer(model_files(f).tok, lean=False).rs is None
wp